In [1]:
import pandas as pd
import numpy as np

In [2]:
# Importamos el archivo json que contiene los datos de los usuarios de streaming
df = pd.read_json('../data/raw/streaming_users_dirty.json')

In [3]:
# 1. Convertir edades ilógicas o fuera de término (menores de 18 o mayores de 100) a NaN
df.loc[(df['age'] < 18) | (df['age'] > 100), 'age'] = np.nan

# 2. Calcular la mediana de edad utilizando solo los registros válidos
mediana_edad = df['age'].median()

# 3. Imputar los valores nulos generados con la mediana calculada
df['age'] = df['age'].fillna(mediana_edad)

# 4. Convertir nuevamente a entero, ya que fillna puede transformar la columna a float
df['age'] = df['age'].astype(int)

# 5. Verificación de la distribución corregida
print(f"Valor utilizado para imputación (Mediana): {mediana_edad}")
print(df['age'].describe())

Valor utilizado para imputación (Mediana): 35.0
count    8160.000000
mean       35.575123
std         9.675783
min        18.000000
25%        29.000000
50%        35.000000
75%        41.000000
max        80.000000
Name: age, dtype: float64


In [4]:
# 1. Normalización de la columna de Países
df['country'] = df['country'].str.strip()

mapeo_paises = {
    'Brazil': 'Brasil', 'brasil': 'Brasil', 'BRA': 'Brasil',
    'colombia': 'Colombia', 'COL': 'Colombia',
    'uruguay': 'Uruguay', 'URY': 'Uruguay',
    'Peru': 'Perú', 'perú': 'Perú', 'PER': 'Perú',
    'chile': 'Chile', 'CHL': 'Chile',
    'argentina': 'Argentina', 'ARG': 'Argentina', 'Argentina ': 'Argentina',
    'méxico': 'México', 'Mexico': 'México', 'MEX': 'México'
}

df['country'] = df['country'].replace(mapeo_paises)


# 2. Normalización de la columna de Géneros
df['favorite_genre'] = df['favorite_genre'].str.strip()

mapeo_generos = {
    'Crime': 'Crimen', 'CRIME': 'Crimen', 'crime': 'Crimen',
    'THRILLER': 'Thriller', 'thriler': 'Thriller', 'Thriller ': 'Thriller',
    'DRAMA': 'Drama', 'drama': 'Drama', 'Drama ': 'Drama',
    'ACCIÓN': 'Acción', 'accion': 'Acción', 'Action': 'Acción',
    'comedy': 'Comedia', 'COMEDIA': 'Comedia', 'Comedia ': 'Comedia',
    'Documentary': 'Documental', 'DOC': 'Documental', 'documental': 'Documental',
    'ROMANCE': 'Romance', 'romance': 'Romance', 'Romance ': 'Romance'
}

df['favorite_genre'] = df['favorite_genre'].replace(mapeo_generos)


# 3. Normalización de la columna de Suscripciones
df['subscription_plan'] = df['subscription_plan'].str.strip()

mapeo_suscripciones = {
    'Std': 'Estándar', 'estandar': 'Estándar', 'STANDARD': 'Estándar',
    'basico': 'Básico', 'básico': 'Básico', 'BASICO': 'Básico', 'Basic': 'Básico',
    'premium': 'Premium', 'Premiun': 'Premium', 'PREMIUM': 'Premium'
}

df['subscription_plan'] = df['subscription_plan'].replace(mapeo_suscripciones)

In [5]:
df = df.drop_duplicates()

Hasta aqui lo que hicimos es normalizar las columnas string y llevar a la media los valores que son anomalos en edad (suponiendo que mayores de 100 por ser demasiado adultos y menores de edad por politicas de contratacion de plataformas de pago, esto ultimo puede variar dependiendo de la plataforma)

In [6]:
# 1. Definimos las columnas numéricas reales de tu dataset
columnas_numericas = ['age', 'monthly_watch_time_mins', 'customer_support_tickets']

# 2. Conteo de negativos totales en el dataset
print("--- Recuento total de valores negativos ---")
for col in columnas_numericas:
    total_negativos = (df[col] < 0).sum()
    print(f"Total en '{col}': {total_negativos} registros")

# 3. Aislamos los duplicados parciales basados exclusivamente en 'user_id'
duplicados_parciales = df[df.duplicated(subset=['user_id'], keep=False)]

# 4. Análisis de superposición: ¿Están los negativos dentro de los duplicados?
print("\n--- Superposición: Negativos en usuarios duplicados ---")
for col in columnas_numericas:
    negativos_en_duplicados = (duplicados_parciales[col] < 0).sum()
    print(f"Negativos en '{col}' que son duplicados: {negativos_en_duplicados}")

# 5. Visualizamos los registros conflictivos ordenados para tu análisis manual
print("\n--- Visualización de pares duplicados por user_id ---")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(duplicados_parciales.sort_values(by=['user_id']))

--- Recuento total de valores negativos ---
Total en 'age': 0 registros
Total en 'monthly_watch_time_mins': 49 registros
Total en 'customer_support_tickets': 29 registros

--- Superposición: Negativos en usuarios duplicados ---
Negativos en 'age' que son duplicados: 0
Negativos en 'monthly_watch_time_mins' que son duplicados: 1
Negativos en 'customer_support_tickets' que son duplicados: 1

--- Visualización de pares duplicados por user_id ---


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,Drama,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1
8126,10721,32,Estándar,959.0,Colombia,Documental,2021-09-02,2
721,10721,32,Estándar,959.0,Colombia,Documental,NaN,2
797,10797,31,Básico,-1.0,México,Comedia,2023-07-01,1
8018,10797,31,Básico,410.4,México,Comedia,2023-07-01,1
1586,11586,35,Básico,1123.6,Brasil,NaN,2019-03-20,1
8154,11586,35,Básico,1123.6,Brasil,Crimen,2019-03-20,1
8055,11746,27,Estándar,891.2,México,Acción,2019-02-26,1
1746,11746,27,Estándar,891.2,México,NaN,2019-02-26,1


In [7]:
# 1. Tratamiento de valores nulos en 'favorite_genre'
# Se asigna "Desconocido" para evitar sesgos analíticos al imputar preferencias arbitrarias
df['favorite_genre'] = df['favorite_genre'].fillna('Desconocido')

# 2. Estandarización de formatos en 'last_login_date'
# 'format=mixed' permite a Pandas inferir los distintos formatos (dd-mm-yyyy, yyyy/mm/dd, etc.)
# 'errors=coerce' convierte cualquier texto completamente ilegible en NaT (Not a Time, el nulo de fechas)
df['last_login_date'] = pd.to_datetime(df['last_login_date'], format='mixed', errors='coerce')

# 3. Imputación de fechas nulas
# Se calcula la fecha mediana (el punto medio cronológico) de los registros válidos
mediana_fechas = df['last_login_date'].median()

# Se imputan los valores NaT con la fecha mediana
df['last_login_date'] = df['last_login_date'].fillna(mediana_fechas)

# Convertir la columna datetime a un formato de texto estándar (AAAA-MM-DD)
# para facilitar su lectura y exportación final a CSV/JSON.
df['last_login_date'] = df['last_login_date'].dt.strftime('%Y-%m-%d')

# 4. Verificación de los cambios
print("Valores nulos restantes en el dataset:")
print(df[['favorite_genre', 'last_login_date']].isnull().sum())
print("\nMuestra de fechas estandarizadas:")
print(df['last_login_date'].head())

Valores nulos restantes en el dataset:
favorite_genre     0
last_login_date    0
dtype: int64

Muestra de fechas estandarizadas:
0    2025-03-04
1    2019-04-02
2    2018-04-13
3    2021-01-31
4    2020-09-30
Name: last_login_date, dtype: str


In [8]:
# 1. Eliminación de duplicados absolutos generados tras las normalizaciones
df = df.drop_duplicates()

# 2. Aislamiento de duplicados parciales basados exclusivamente en 'user_id'
duplicados_parciales = df[df.duplicated(subset=['user_id'], keep=False)]

# 3. Visualización de los pares conflictivos para análisis manual
print(f"Total de filas en conflicto por user_id: {duplicados_parciales.shape[0]}")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(duplicados_parciales.sort_values(by=['user_id']))

Total de filas en conflicto por user_id: 36


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,Drama,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1
8126,10721,32,Estándar,959.0,Colombia,Documental,2021-09-02,2
721,10721,32,Estándar,959.0,Colombia,Documental,2022-02-17,2
797,10797,31,Básico,-1.0,México,Comedia,2023-07-01,1
8018,10797,31,Básico,410.4,México,Comedia,2023-07-01,1
1586,11586,35,Básico,1123.6,Brasil,Desconocido,2019-03-20,1
8154,11586,35,Básico,1123.6,Brasil,Crimen,2019-03-20,1
8055,11746,27,Estándar,891.2,México,Acción,2019-02-26,1
1746,11746,27,Estándar,891.2,México,Desconocido,2019-02-26,1


In [9]:
# 1. Eliminación de duplicados absolutos generados tras las normalizaciones previas
df = df.drop_duplicates()

# 2. Creación de banderas booleanas para identificar "mala calidad" en el registro.
# True indica la presencia de un defecto. False indica un dato limpio.
df['_tiene_negativo'] = (df['monthly_watch_time_mins'] < 0) | (df['customer_support_tickets'] < 0)
df['_tiene_desconocido'] = (df['favorite_genre'] == 'Desconocido')
df['_tiene_nulos'] = df.isnull().any(axis=1)

# Validación de espacios en extremos (aunque mitigado por .str.strip() previo).
df['_tiene_espacios'] = df['subscription_plan'].astype(str).str.contains(r'^\s|\s$', regex=True) | \
                        df['country'].astype(str).str.contains(r'^\s|\s$', regex=True)

# 3. Ordenamiento del DataFrame.
# Al ordenar valores booleanos, False (0) va antes que True (1).
# Esto asegura que para un mismo 'user_id', la fila con menos defectos quede arriba.
df = df.sort_values(by=[
    'user_id', 
    '_tiene_negativo', 
    '_tiene_desconocido', 
    '_tiene_nulos', 
    '_tiene_espacios'
])

# 4. Eliminación de duplicados parciales.
# Al conservar el 'first', nos quedamos con el registro de mayor calidad y borramos los defectuosos.
df = df.drop_duplicates(subset=['user_id'], keep='first')

# 5. Eliminación de las columnas auxiliares creadas para el ordenamiento.
columnas_temporales = ['_tiene_negativo', '_tiene_desconocido', '_tiene_nulos', '_tiene_espacios']
df = df.drop(columns=columnas_temporales)

# 6. Verificación final de conflictos.
duplicados_parciales_restantes = df[df.duplicated(subset=['user_id'], keep=False)]
print(f"Total de filas en conflicto por user_id tras aplicar las reglas lógicas: {duplicados_parciales_restantes.shape[0]}")

# 7. Aislamiento de duplicados parciales basados exclusivamente en 'user_id'
duplicados_parciales = df[df.duplicated(subset=['user_id'], keep=False)]

# 8. Visualización de los pares conflictivos para análisis manual
print(f"Total de filas en conflicto por user_id: {duplicados_parciales.shape[0]}")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(duplicados_parciales.sort_values(by=['user_id']))

Total de filas en conflicto por user_id tras aplicar las reglas lógicas: 0
Total de filas en conflicto por user_id: 0


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets


Hasta aqui logramos eliminar todos los duplicados o ID duplicados

In [10]:
# Definimos las columnas numéricas pendientes de revisión
columnas_numericas = ['monthly_watch_time_mins', 'customer_support_tickets']

# Aplicamos una máscara booleana: filtra las filas donde ALGUNA (any) de estas columnas sea menor a 0
# axis=1 indica que la evaluación de la condición (< 0) se realiza fila por fila
filas_con_negativos = df[(df[columnas_numericas] < 0).any(axis=1)]

# Visualizamos el resultado sin truncamiento
print(f"Total de registros únicos remanentes con valores negativos: {filas_con_negativos.shape[0]}")
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(filas_con_negativos)

Total de registros únicos remanentes con valores negativos: 76


,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
378,10378,58,Básico,577.3,México,Romance,2018-10-09,-1
443,10443,31,Estándar,-120.0,Chile,Thriller,2019-06-07,1
682,10682,54,Premium,NaN,Uruguay,Acción,2021-12-11,-1
1099,11099,35,Básico,771.3,Perú,Romance,2021-03-25,-1
1126,11126,35,Premium,-1.0,Colombia,Documental,2021-01-24,1
1186,11186,35,Premium,-120.0,Brasil,Drama,2022-12-09,0
1340,11340,40,Estándar,-120.0,Perú,Documental,2023-05-28,2
1405,11405,28,Básico,535.7,Colombia,Thriller,2019-09-25,-1
1466,11466,35,Estándar,-120.0,Perú,Acción,2018-03-02,1
1719,11719,35,Básico,1017.7,Argentina,Documental,2018-12-27,-1


In [11]:
# Definimos las columnas a procesar
columnas_metricas = ['monthly_watch_time_mins', 'customer_support_tickets']

# 1. Transformación de negativos a positivos mediante valor absoluto
df[columnas_metricas] = df[columnas_metricas].abs()

# 2. Imputación de nulos con valor 0
df[columnas_metricas] = df[columnas_metricas].fillna(0)

# 3. Verificación del impacto
print("--- Estadísticas tras la corrección ---")
print(df[columnas_metricas].describe())

--- Estadísticas tras la corrección ---
       monthly_watch_time_mins  customer_support_tickets
count              8000.000000               8000.000000
mean               1087.156250                  1.827625
std                5301.597778                 11.445121
min                   0.000000                  0.000000
25%                 466.000000                  0.000000
50%                 742.200000                  1.000000
75%                1035.425000                  1.000000
max               99999.000000                150.000000


In [12]:
# 1. Eliminación de duplicados absolutos
df = df.drop_duplicates()

# 2. RESOLUCIÓN DE DUPLICADOS PARCIALES (Selección de supervivencia)
df['_tiene_negativo'] = (df['monthly_watch_time_mins'] < 0) | (df['customer_support_tickets'] < 0)
df['_tiene_desconocido'] = (df['favorite_genre'] == 'Desconocido')
df['_tiene_nulos'] = df.isnull().any(axis=1)
df['_tiene_espacios'] = df['subscription_plan'].astype(str).str.contains(r'^\s|\s$', regex=True) | \
                        df['country'].astype(str).str.contains(r'^\s|\s$', regex=True)

df = df.sort_values(by=['user_id', '_tiene_negativo', '_tiene_desconocido', '_tiene_nulos', '_tiene_espacios'])
df = df.drop_duplicates(subset=['user_id'], keep='first')
df = df.drop(columns=['_tiene_negativo', '_tiene_desconocido', '_tiene_nulos', '_tiene_espacios'])

# 3. TRATAMIENTO DE ANOMALÍAS EN REGISTROS ÚNICOS
columnas_metricas = ['monthly_watch_time_mins', 'customer_support_tickets']

# Convertir negativos a absolutos
df[columnas_metricas] = df[columnas_metricas].abs()

# Truncamiento de Outliers Extremos
# Límite físico para minutos (44640) y límite operativo lógico para tickets (20)
df.loc[df['monthly_watch_time_mins'] > 44640, 'monthly_watch_time_mins'] = np.nan
df.loc[df['customer_support_tickets'] > 20, 'customer_support_tickets'] = np.nan

# 4. IMPUTACIÓN
# Aplicando su regla de negocio: asignar 0 a los valores nulos en estas métricas
df[columnas_metricas] = df[columnas_metricas].fillna(0)

# 5. VERIFICACIÓN FINAL
print("--- Estadísticas Finales Consolidadas ---")
print(df[columnas_metricas].describe())

--- Estadísticas Finales Consolidadas ---
       monthly_watch_time_mins  customer_support_tickets
count              8000.000000               8000.000000
mean                768.408750                  0.794500
std                 507.852902                  0.896728
min                   0.000000                  0.000000
25%                 460.500000                  0.000000
50%                 737.100000                  1.000000
75%                1029.000000                  1.000000
max                4193.700000                  5.000000


In [13]:
# Exportar el dataset limpio para su uso en el EDA
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
print("Dataset exportado exitosamente con 8000 registros y estructura limpia.")

Dataset exportado exitosamente con 8000 registros y estructura limpia.
